In [1]:
import urllib.request
import json
import pandas as pd
import os

# Ruta donde se guardará el archivo JSON
ruta_archivo = '../data/exercises.json'

# Descargar el dataset automáticamente si no existe localmente
url_dataset = 'https://raw.githubusercontent.com/hasaneyldrm/exercises-dataset/main/data/exercises.json'

if not os.path.exists(ruta_archivo):
    print("Descargando el dataset desde GitHub...")
    urllib.request.urlretrieve(url_dataset, ruta_archivo)
    print("¡Descarga completada!")

# Cargar el JSON a la memoria
with open(ruta_archivo, 'r', encoding='utf-8') as f:
    datos = json.load(f)

# Convertir a DataFrame de Pandas
df = pd.DataFrame(datos)

# Mostrar información básica
print(f"\nTotal de registros cargados: {len(df)}")
display(df.head(3))
df.info()

Descargando el dataset desde GitHub...
¡Descarga completada!

Total de registros cargados: 1324


,id,name,category,body_part,equipment,instructions,instruction_steps,muscle_group,secondary_muscles,target,image,gif_url,media_id,created_at,attribution
0,0001,3/4 sit-up,waist,waist,body weight,{'en': 'Lie flat on your back with your knees ...,{'en': ['Lie flat on your back with your knees...,hip flexors,"[hip flexors, lower back]",abs,images/0001-2gPfomN.jpg,videos/0001-2gPfomN.gif,2gPfomN,2026-03-18T12:31:32.854798+00:00,© Gym visual — https://gymvisual.com/
1,0002,45° side bend,waist,waist,body weight,{'en': 'Stand with your feet shoulder-width ap...,{'en': ['Stand with your feet shoulder-width a...,obliques,[obliques],abs,images/0002-Hy9D21L.jpg,videos/0002-Hy9D21L.gif,Hy9D21L,2026-03-18T12:31:32.854953+00:00,© Gym visual — https://gymvisual.com/
2,0003,air bike,waist,waist,body weight,{'en': 'Lie flat on your back with your hands ...,{'en': ['Lie flat on your back with your hands...,hip flexors,[hip flexors],abs,images/0003-1ZFqTDN.jpg,videos/0003-1ZFqTDN.gif,1ZFqTDN,2026-03-18T12:31:32.854977+00:00,© Gym visual — https://gymvisual.com/


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1324 entries, 0 to 1323
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   id                 1324 non-null   object
 1   name               1324 non-null   object
 2   category           1324 non-null   object
 3   body_part          1324 non-null   object
 4   equipment          1324 non-null   object
 5   instructions       1324 non-null   object
 6   instruction_steps  1324 non-null   object
 7   muscle_group       1324 non-null   object
 8   secondary_muscles  1324 non-null   object
 9   target             1324 non-null   object
 10  image              1324 non-null   object
 11  gif_url            1324 non-null   object
 12  media_id           1324 non-null   object
 13  created_at         1324 non-null   object
 14  attribution        1324 non-null   object
dtypes: object(15)
memory usage: 155.3+ KB


In [12]:
import pandas as pd
import json

# 1. Cargar los datos originales
ruta_archivo = '../data/exercises.json'
with open(ruta_archivo, 'r', encoding='utf-8') as f:
    datos = json.load(f)
df = pd.DataFrame(datos)

# 2. Función para contar los pasos de instrucciones en español
def contar_instrucciones_es(x):
    if isinstance(x, dict):
        pasos_es = x.get('es')
        if isinstance(pasos_es, list):
            return len(pasos_es)
        # si no hay versión en español, probamos con inglés como respaldo
        pasos_en = x.get('en')
        if isinstance(pasos_en, list):
            return len(pasos_en)
    return 0

# 3. Función para contar músculos involucrados (primario + secundarios)
def contar_musculos(muscle_group, secondary_muscles):
    total = 0
    # muscle_group es un string único -> cuenta como 1 si no está vacío
    if isinstance(muscle_group, str) and muscle_group.strip():
        total += 1
    # secondary_muscles ya es una lista
    if isinstance(secondary_muscles, list):
        total += len(secondary_muscles)
    return total

# 4. Feature Engineering
if 'instruction_steps' in df.columns:
    df['num_instrucciones'] = df['instruction_steps'].apply(contar_instrucciones_es)
else:
    df['num_instrucciones'] = 0

df['num_musculos_involucrados'] = df.apply(
    lambda fila: contar_musculos(fila.get('muscle_group'), fila.get('secondary_muscles')),
    axis=1
)

# 5. Extraer también el texto de instrucciones y el nombre del músculo primario en español (opcional pero útil para el EDA)
def extraer_texto_es(x):
    if isinstance(x, dict):
        return x.get('es', x.get('en', ''))
    return ''

df['instrucciones_texto_es'] = df['instructions'].apply(extraer_texto_es) if 'instructions' in df.columns else ''
df['musculo_primario'] = df['muscle_group'] if 'muscle_group' in df.columns else ''

# 6. Guardar el dataset limpio
df.to_csv('../data/exercises_limpio.csv', index=False)

# 7. Verificación
columnas_mostrar = ['name', 'num_instrucciones', 'num_musculos_involucrados', 'musculo_primario']
columnas_mostrar = [col for col in columnas_mostrar if col in df.columns]

print("\n¡Datos limpios listos para el EDA de tu compañero!")
display(df[columnas_mostrar].head())

print("\n=== Verificación final ===")
print("Total instrucciones:", df['num_instrucciones'].sum())
print("Total músculos:", df['num_musculos_involucrados'].sum())


¡Datos limpios listos para el EDA de tu compañero!


,name,num_instrucciones,num_musculos_involucrados,musculo_primario
0,3/4 sit-up,5,3,hip flexors
1,45° side bend,5,2,obliques
2,air bike,5,2,hip flexors
3,all fours squad stretch,5,3,hamstrings
4,alternate heel touchers,5,2,obliques



=== Verificación final ===
Total instrucciones: 7710
Total músculos: 3905
